## Objective: 

## Use Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset

## STEP 1: Importing Libraries 

In [1]:
import sqlite3
import pandas as pd
print("All libraries imported successfully")

All libraries imported successfully


## STEP 2: Connect Database

In [2]:
conn=sqlite3.connect("superstore_sales_analysis")
cursor=conn.cursor()
print("Database connection successful")

Database connection successful


# Step 3: Setup Data

### Step 3.1:Load the dataset

In [3]:
#Load Dataset
df = pd.read_csv("Superstores.csv", encoding="latin1")
print(f"Shape is:{df.shape}")
print(f"Rows are:{df.shape[0]}")
print(f"Columns are:{df.shape[1]}")

Shape is:(9994, 21)
Rows are:9994
Columns are:21


In [4]:
# Creating table superstore
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


### Step 3.2: Creating Required Tables

### Table 1: Customer Table

In [20]:
cursor.execute('''
create table customers as
select distinct
    "Customer ID" AS cust_id,
    "Customer Name" AS cust_name,
    Segment
from superstore_raw''')
print("Table Created Successfully")
conn.commit()

Table Created Successfully


### Table 2:Products Table

In [22]:
cursor.execute('''
create table products as
select distinct
    "Product ID" AS product_id,
    "Product Name" AS product_name,
    Category,
    "Sub-Category" AS sub_category
FROM superstore_raw''')
print("Table Created Successfully")
conn.commit()

Table Created Successfully


### Table 3:Orders Table

In [23]:
cursor.execute('''
create table orders as
select distinct
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    "Ship Date" AS ship_date,
    "Ship Mode" AS ship_mode,
    "Customer ID" AS customer_id,
    "Product ID" AS product_id,
    Sales,
    Quantity,
    Discount,
    Profit
from superstore_raw''')
print("Table Created Successfully")
conn.commit()

Table Created Successfully


In [24]:
# Query Execution Function
def run_query(query):
    return pd.read_sql_query(query, conn)
print("Function created successfully")

Function created successfully


# Step 4:Perform Required Queries


In [95]:
# Q1. Find all orders where sales are greater than the average sales
q1='''
select * from orders
where Sales>(select avg(Sales) from orders)
'''
df_q1=run_query(q1)
df_q1

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...,...,...
2354,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10004080,271.9600,5,0.20,27.1960
2355,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10002496,249.5840,2,0.20,31.1980
2356,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,OFF-BI-10002026,437.4720,14,0.20,153.1152
2357,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,TEC-PH-10003645,258.5760,2,0.20,19.3932


In [121]:
print("Insight:Identifies orders generating revenue above the overall average sales value")

Insight:Identifies orders generating revenue above the overall average sales value


In [99]:
# Q2. Find the highest sales order for each customer
q2='''select * from orders o
where sales=(select max(Sales) from orders
where customer_id=o.customer_id)
order by Sales desc
'''
df_q2=run_query(q2)
df_q2

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2014-145317,3/18/2014,3/23/2014,Standard Class,SM-20320,TEC-MA-10002412,22638.480,6,0.5,-1811.0784
1,CA-2016-118689,10/2/2016,10/9/2016,Standard Class,TC-20980,TEC-CO-10004722,17499.950,5,0.0,8399.9760
2,CA-2017-140151,3/23/2017,3/25/2017,First Class,RB-19360,TEC-CO-10004722,13999.960,4,0.0,6719.9808
3,CA-2017-127180,10/22/2017,10/24/2017,First Class,TA-21385,TEC-CO-10004722,11199.968,4,0.2,3919.9888
4,CA-2017-166709,11/17/2017,11/22/2017,Standard Class,HL-15040,TEC-CO-10004722,10499.970,3,0.0,5039.9856
...,...,...,...,...,...,...,...,...,...,...
790,CA-2016-163951,12/30/2016,1/2/2017,First Class,CJ-11875,OFF-AR-10004269,16.520,5,0.2,1.6520
791,CA-2017-115070,4/10/2017,4/14/2017,Second Class,MG-18205,FUR-FU-10003829,12.320,5,0.2,1.8480
792,CA-2016-107475,6/7/2016,6/11/2016,Standard Class,RS-19870,OFF-FA-10002988,9.648,6,0.2,3.4974
793,CA-2016-152331,6/26/2016,6/30/2016,Standard Class,LD-16855,OFF-AR-10001547,5.304,3,0.2,0.4641


In [122]:
print("Insight:Finds the highest-value order placed by each customer")

Insight:Finds the highest-value order placed by each customer


In [97]:
# Q3:Calculate total sales for each customer
q3='''
with total_sale as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select * from total_sale
'''
df_q3=run_query(q3)
df_q3

,customer_id,Total_Sale
0,AA-10315,5563.560
1,AA-10375,1056.390
2,AA-10480,1790.512
3,AA-10645,5086.935
4,AB-10015,886.156
...,...,...
788,XP-21865,2374.658
789,YC-21895,5454.350
790,YS-21880,6720.444
791,ZC-21910,8025.707


In [123]:
print("Insight:Calculates total sales contributed by every customer")

Insight:Calculates total sales contributed by every customer


In [124]:
# Q4:Find customers whose total sales are above average
q4='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
),
 customer_average_sale as(
select avg(Total_Sale) as Average_Sale from customer_sales
)
select * from
customer_sales
where Total_Sale>(select Average_Sale from customer_average_sale)
'''
df_q4=run_query(q4)
df_q4

,customer_id,Total_Sale
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


In [102]:
print("Insight:Identifies customers whose total sales exceed the average customer sales")

Insight:Identifies customers whose total sales exceed the average customer sales


In [103]:
# Q5:Rank all customers based on total sales 
q5='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select * , dense_rank() over(order by Total_Sale desc) as rn from customer_sales
'''
df_q5=run_query(q5)
df_q5

,customer_id,Total_Sale,rn
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


In [125]:
print("Insight:Ranks customers based on their total sales contribution")


Insight:Ranks customers based on their total sales contribution


In [105]:
# Q6:Assign row numbers to each order within a customer. 
q6='''select*,
       row_number() over
       (
           partition by customer_id
           order by order_date
       ) as row_num
from orders '''
df_q6=run_query(q6)
df_q6

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit,row_num
0,CA-2015-121391,10/4/2015,10/7/2015,First Class,AA-10315,OFF-ST-10001590,26.960,2,0.0,7.0096,1
1,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,OFF-SU-10000151,3930.072,3,0.2,-786.0144,2
2,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,OFF-FA-10001332,2.304,1,0.2,0.7776,3
3,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,TEC-PH-10000895,431.976,3,0.2,32.3982,4
4,CA-2016-103982,3/3/2016,3/8/2016,Standard Class,AA-10315,TEC-AC-10002857,41.720,7,0.2,5.7365,5
...,...,...,...,...,...,...,...,...,...,...,...
9988,CA-2016-152471,7/8/2016,7/8/2016,Same Day,ZD-21925,TEC-PH-10002824,823.960,5,0.2,51.4975,5
9989,CA-2016-152471,7/8/2016,7/8/2016,Same Day,ZD-21925,OFF-PA-10004965,15.984,2,0.2,4.9950,6
9990,CA-2014-143336,8/27/2014,9/1/2014,Second Class,ZD-21925,OFF-AR-10003056,8.560,2,0.0,2.4824,7
9991,CA-2014-143336,8/27/2014,9/1/2014,Second Class,ZD-21925,TEC-PH-10001949,213.480,3,0.2,16.0110,8


In [106]:
print("Insight:Assigns a purchase sequence number to each customer's orders")

Insight:Assigns a purchase sequence number to each customer's orders


In [107]:
# Q7:Display top 3 customers based on total sales
q7='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select * from(select *,dense_rank()over(order by Total_Sale desc) as customer_rank from customer_sales) x
where customer_rank<4
'''
df_q7=run_query(q7)
df_q7

,customer_id,Total_Sale,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


In [108]:
print("Insight:Highlights the top 3 revenue-generating customers")

Insight:Highlights the top 3 revenue-generating customers


# Final Combined Query 

In [109]:
# Write one final query that shows: 
#Customer Name  
#Total Sales  
#Rank
q8='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select c.cust_name,cs.Total_Sale,dense_rank()over(order by cs.Total_Sale desc) as customer_rank from
customers c
inner join
customer_sales cs
on cs.customer_id=c.cust_id
'''
df_q8=run_query(q8)
df_q8


,cust_name,Total_Sale,customer_rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,2
2,Raymond Buch,15117.339,3
3,Tom Ashbrook,14595.620,4
4,Adrian Barton,14473.571,5
...,...,...,...
788,Roy Skaria,22.328,789
789,Mitch Gastineau,16.739,790
790,Carl Jackson,16.520,791
791,Lela Donovan,5.304,792


In [110]:
print("Insight:Provides a complete view of customer sales performance and ranking")

Insight:Provides a complete view of customer sales performance and ranking


# Mini Project: Customer Sales Insights

## Q1:Who are the top 5 customers?  

In [111]:
q9='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select cs.customer_id,c.cust_name,cs.Total_Sale ,dense_rank()over(order by cs.Total_Sale desc) as customer_rank 
from
customer_sales cs
inner join 
customers c
on cs.customer_id=c.cust_id
limit 5;
'''

df_q9=run_query(q9)
print("Top 5 Customers")
df_q9.style.hide(axis="index")

Top 5 Customers


customer_id,cust_name,Total_Sale,customer_rank
SM-20320,Sean Miller,25043.050000,1
TC-20980,Tamara Chand,19052.218000,2
RB-19360,Raymond Buch,15117.339000,3
TA-21385,Tom Ashbrook,14595.620000,4
AB-10105,Adrian Barton,14473.571000,5


In [112]:
print("Insight:Identifies the top 5 customers contributing the most revenue")

Insight:Identifies the top 5 customers contributing the most revenue


## Q2:Who are the bottom 5 customers?  

In [113]:
q10='''
with customer_sales as (
select customer_id,sum(Sales) Total_Sale from orders
group by customer_id
)
select cs.customer_id,c.cust_name,cs.Total_Sale ,dense_rank()over(order by cs.Total_Sale desc) as customer_rank 
from
customer_sales cs
inner join 
customers c
on cs.customer_id=c.cust_id
order by customer_rank desc
limit 5;
'''

df_q10=run_query(q10)
print("Bottom 5 Customers")
df_q10.style.hide(axis="index")

Bottom 5 Customers


customer_id,cust_name,Total_Sale,customer_rank
TS-21085,Thais Sissman,4.833000,793
LD-16855,Lela Donovan,5.304000,792
CJ-11875,Carl Jackson,16.520000,791
MG-18205,Mitch Gastineau,16.739000,790
RS-19870,Roy Skaria,22.328000,789


In [114]:
print("Insight:Identifies the bottom 5 customers with the lowest sales contribution")

Insight:Identifies the bottom 5 customers with the lowest sales contribution


## Q3:Which customers made only one order?  

In [115]:
q11='''
with customer_count as (
select customer_id,count(*) as order_count from orders
group by customer_id
having order_count=1
)
select c.cust_name,cs.customer_id,cs.order_count from 
customer_count cs
inner join
customers c
on cs.customer_id=c.cust_id

'''
df_q11=run_query(q11)
print("Customer who made only 1 order")
df_q11.style.hide(axis="index")

Customer who made only 1 order


cust_name,customer_id,order_count
Jocasta Rupert,JR-15700,1
Anthony O'Donnell,AO-10810,1
Lela Donovan,LD-16855,1
Carl Jackson,CJ-11875,1
Ricardo Emerson,RE-19405,1


In [116]:
print("Insight:Finds customers who made only one purchase, indicating possible churn")

Insight:Finds customers who made only one purchase, indicating possible churn


## Q4:Which customers have above-average sales?  

In [117]:
q12='''
with customer_sales as
(
    select
        customer_id,
        sum(Sales) as total_sales
    from orders
    group by customer_id
)

select *
from customer_sales
where total_sales >
(
    select avg(total_sales)
    from customer_sales
)'''
df_q12=run_query(q12)
print("Customer have above average sales")
df_q12.style.hide(axis="index")

Customer have above average sales


customer_id,total_sales
AA-10315,5563.560000
AA-10645,5086.935000
AB-10060,7755.620000
AB-10105,14473.571000
AC-10450,5527.846000
AD-10180,6106.880000
AG-10675,3489.039600
AG-10900,4510.797000
AH-10075,3250.337000
AH-10210,4805.344000


In [118]:
print("Insight:Highlights customers generating above-average sales")

Insight:Highlights customers generating above-average sales


## Q5:What is the highest order value per customer? 

In [119]:
q13='''
with max_values as(
select customer_id,max(Sales) as highest_value from orders
group by customer_id
)
select * from max_values
order by highest_value desc
'''
df_q13=run_query(q13)
print("Highest order value per customer")
df_q13

Highest order value per customer


,customer_id,highest_value
0,SM-20320,22638.480
1,TC-20980,17499.950
2,RB-19360,13999.960
3,TA-21385,11199.968
4,HL-15040,10499.970
...,...,...
788,CJ-11875,16.520
789,MG-18205,12.320
790,RS-19870,9.648
791,LD-16855,5.304


In [120]:
print("Insight:Shows the maximum order value achieved by each customer") 

Insight:Shows the maximum order value achieved by each customer
